In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

In [ ]:
from config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

geolocation = spark.read.csv(
    s3_path("bronze", "geolocation", "olist_geolocation_dataset.csv"),
    header=True,
    inferSchema=True
)

geolocation.show(5)

In [ ]:
geolocation.printSchema()

print(f"Number of records: {geolocation.count()}")

geolocation.show(10, truncate=False)

geolocation.describe().show()

from pyspark.sql.functions import col, count, when

geolocation.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in geolocation.columns
]).show()

In [ ]:
total = geolocation.count()

distinct = geolocation.distinct().count()

print("Total Rows:", total)
print("Distinct Rows:", distinct)
print("Duplicate Rows:", total - distinct)

In [ ]:
from pyspark.sql.functions import countDistinct

geolocation.groupBy("geolocation_zip_code_prefix") \
    .agg(
        countDistinct("geolocation_city").alias("city_count"),
        countDistinct("geolocation_state").alias("state_count")
    ) \
    .filter(
        (col("city_count") > 1) |
        (col("state_count") > 1)
    ) \
    .show(20, truncate=False)

In [ ]:
geolocation.select("geolocation_city").distinct().count()

In [ ]:
geolocation.select("geolocation_state") \
    .distinct() \
    .orderBy("geolocation_state") \
    .show(30, False)

In [ ]:
geolocation_clean = geolocation.dropDuplicates()